In [ ]:
import dataiku
import pandas as pd

In [0]:
built_in_plugins = ['builtin-macros', 'local-r-dev-setup', 'default-samples', 'code-studio-blocks',
                   'colorbrewer-palettes', 'k8s-metrics-utils', 'eks-clusters', 'project-standards']

In [0]:
# Connect to Dataiku DSS
client = dataiku.api_client()

In [ ]:
plugins = client.list_plugins()

results = []

for plugin_info in plugins:
    plugin_id = plugin_info['id']
    plugin = client.get_plugin(plugin_id)
    usages = plugin.list_usages().usages

    results.append({
        "Plugin ID": plugin_id,
        "Usage Count": len(usages),
    })

In [0]:
df_results = pd.DataFrame(results)
df_results.sort_values(by='Usage Count', ascending=False)

In [0]:
plugins_to_remove = set(df_results.loc[df_results["Usage Count"] == 0, "Plugin ID"].astype(str).tolist()) - set(built_in_plugins)
plugins_to_remove

In [0]:
for plugin_id_to_remove in plugins_to_remove:
    plugin = client.get_plugin(plugin_id_to_remove)
    print(f"Uninstalling plugin because it has 0 usages and is not built in: {plugin_id_to_remove}")
    future = plugin.delete(force=False)
    future.wait_for_result()